In [24]:
import shutil
from pathlib import Path
import json

import polars as pl

In [26]:
results = pl.read_ndjson("results.jsonl").unnest("response")
results.head()

source_file,criterion,study_result,explanation,study_contradicts_rubric,error,raw_response
str,str,str,str,bool,str,str
"""1956b7b9-0b75-43a4-8ef6-e4f726…","""Includes information that some…","""We identified 1,096 patients a…","""The study result describes an …",false,"""""",""""""
"""e5725a9f-859f-4ec7-82f2-91c0cc…","""Clarifies that iris prolapse i…","""------------------------------…","""The rubric specifically clarif…",false,"""""",""""""
"""61b90306-ccb5-42b7-89d2-b58dcf…","""Describes that although some p…","""This cohort study investigated…","""The rubric criterion discusses…",false,"""""",""""""
"""76092487-d17f-4bfd-825f-333362…","""Mentions that living kidney do…","""We identified patients who had…","""The rubric criterion describes…",false,"""""",""""""
"""fd59b9c2-4750-453a-a3de-a99522…","""Correctly states that the mana…","""This cohort study examined tim…","""The rubric focuses on tailorin…",false,"""""",""""""


In [27]:
errors = results.filter(pl.col("error") != "")
len(errors)

0

In [28]:
results.filter(pl.col("study_contradicts_rubric"))

source_file,criterion,study_result,explanation,study_contradicts_rubric,error,raw_response
str,str,str,str,bool,str,str
"""b10b5474-861a-461e-93ef-167ab7…","""Provides links to the source o…","""see report This case includes…","""The rubric criterion mandates …",true,"""""",""""""
"""47c18c3f-4ced-4e61-b070-67c614…","""Describes the methods of inves…","""We identified patients who had…","""The rubric criterion requires …",true,"""""",""""""
"""e2ab624a-a05d-4101-9c88-eceef7…","""Fills out fabricated sedation …","""We analyzed patients at Stanfo…","""The rubric criterion requires …",true,"""""",""""""
"""b2f4a7e9-7399-43d1-8b73-b3822a…","""Gives definitive advice for or…","""We identified 2528 patients 75…","""The rubric expects a straightf…",true,"""""",""""""
"""70c10085-1386-441f-81f5-4d21cf…","""Instructs the user to give the…","""We analyzed all children (aged…","""The rubric criterion directs t…",true,"""""",""""""
…,…,…,…,…,…,…
"""d8cbf5b5-ae23-4a65-821f-a847a8…","""FERGIcor trials date back to 2…","""This cohort study examined tim…","""The criterion specifies that s…",true,"""""",""""""
"""3d6c6016-d082-46e0-abad-d89be4…","""Suggests that the Ottawa ankle…","""We identified adult patients a…","""The rubric criterion focuses e…",true,"""""",""""""
"""1cbd2c6c-09fd-46cc-bd66-c3b96e…","""Presents only positive outcome…","""In a case series from 2017 to …","""The rubric instructs to presen…",true,"""""",""""""


In [ ]:
input_dir = Path(
    "/Users/max/Developer/repos/HealthBench/src/simple_evals/improvement/examples_of_bad_static_criteria/opensearch/to_ingest"
)
contradictions_dir = Path("contradictions")
for row in results.filter(pl.col("study_contradicts_rubric")).iter_rows(named=True):
    input_file = input_dir / row["source_file"]
    output_file = contradictions_dir / row["source_file"]
    out_data = json.loads(input_file.read_text())
    out_data["explanation"] = row["explanation"]
    output_file.write_text(json.dumps(out_data, indent=2))